# EXP-03C — Diagnóstico mínimo de fonte histórica Ethereum

**Status: preparado, não executado.** Antes de consultas em lote, validar 1 bloco histórico com referência WETH conhecida. Não usar resultados como rótulos, nem retreinar modelo. Nenhum segredo é salvo no Drive ou nos relatórios.

**Importante:** requer endpoint HTTPS de um provedor de archive RPC (Infura, Alchemy ou equivalente). Não é possível usar o mesmo endpoint público que falhou no EXP-03B como prova de capacidade histórica.

In [ ]:
from getpass import getpass
import os
endpoint = getpass("Cole seu endpoint HTTPS archive RPC (valor oculto): ").strip()
assert endpoint.startswith("https://"), "Use endpoint HTTPS"
os.environ["ETHEREUM_ARCHIVE_RPC"] = endpoint

### Prova de estado histórico
A rotina primeiro consulta bytecode do contrato WETH no bloco **24.236.673** (controle positivo de disponibilidade histórica). Somente se retornar bytecode não vazio consulta **um** dos seis controles do EXP-03, no mesmo bloco. O retorno pode ser `NO_GO_ARCHIVE`, `NO_GO_CONTROL` ou `PILOT_RPC_OK`. Não expõe a URL/token em outputs.

In [ ]:
"""EXP-03C pilot: confirm ability of authenticated archive RPC to return old state.
No model inference. No labels. Minimal archive probe; tokens never written to outputs.
"""
import hashlib,json,os,time,urllib.request,urllib.error
from datetime import datetime,timezone
from pathlib import Path

BLOCK=24236673
REFERENCE='0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2' # WETH, known deployed long before 2026
CONTROL='0xbd70bb67c90ea7d01abcf5b0c973a3830c7cff88'

def rpc(url,method,params):
    payload=json.dumps({'jsonrpc':'2.0','id':1,'method':method,'params':params}).encode()
    request=urllib.request.Request(url,data=payload,headers={'Content-Type':'application/json','User-Agent':'ethmix-research-archive-probe/1.0'},method='POST')
    try:
        with urllib.request.urlopen(request,timeout=25) as r:
            return {'http':r.status,'data':json.load(r),'error':None}
    except urllib.error.HTTPError as e: return {'http':e.code,'data':None,'error':'HTTP_'+str(e.code)}
    except Exception as e: return {'http':None,'data':None,'error':type(e).__name__}

def summarize(item):
    d=item.get('data') or {};result=d.get('result');code=result if isinstance(result,str) and result.startswith('0x') else None
    return {'http_status':item.get('http'),'rpc_error':d.get('error') if isinstance(d,dict) else item.get('error'),
            'valid_hex_response':code is not None,'code_bytes':(len(code)-2)//2 if code else None,
            'code_sha256':hashlib.sha256(bytes.fromhex(code[2:])).hexdigest() if code is not None else None,
            'state':'code_present' if code is not None and len(code)>2 else ('code_empty' if code=='0x' else 'inconclusive')}

def main():
    url=os.getenv('ETHEREUM_ARCHIVE_RPC','').strip()
    if not url.startswith('https://'):raise RuntimeError('Configure ETHEREUM_ARCHIVE_RPC como HTTPS autenticado.')
    directory=Path('/content/EXP03C_archive_probe')
    if directory.exists() and list(directory.iterdir()):raise RuntimeError('A pasta de saída já existe e não está vazia: não sobrescrever.')
    # Confirm Ethereum mainnet before any historical state read.
    chain_id_response=rpc(url,'eth_chainId',[])
    chain_id=(chain_id_response.get('data') or {}).get('result')
    if chain_id!='0x1':
        directory.mkdir(parents=True,exist_ok=True)
        result={'utc':datetime.now(timezone.utc).isoformat(),'chain_id':chain_id,
                'decision':'NO_GO_WRONG_NETWORK','notice':'Endpoint nao comprovou chainId 1; nenhum endereco historico consultado.'}
        out=directory/'diagnostico_archive.json'
        out.write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding='utf8')
        print(json.dumps(result,ensure_ascii=False,indent=2))
        return result
    chain=summarize(rpc(url,'eth_getCode',[REFERENCE,hex(BLOCK)]))
    if chain['state'] != 'code_present':
        result={'utc':datetime.now(timezone.utc).isoformat(),'anchor_block':BLOCK,'reference':REFERENCE,
                'reference_check':chain,'decision':'NO_GO_ARCHIVE','notice':'Provedor nao comprovou codigo historico na referencia. Nenhum controle consultado.'}
    else:
        time.sleep(0.5)
        check=summarize(rpc(url,'eth_getCode',[CONTROL,hex(BLOCK)]))
        result={'utc':datetime.now(timezone.utc).isoformat(),'anchor_block':BLOCK,'reference':REFERENCE,
                'reference_check':chain,'control':CONTROL,'control_check':check,
                'decision':'PILOT_RPC_OK' if check['state']!='inconclusive' else 'NO_GO_CONTROL',
                'notice':'Uma resposta de codigo no bloco nao determina funcao nem rotulo cientifico.'}
    directory.mkdir(parents=True,exist_ok=True)
    out=directory/'diagnostico_archive.json'
    out.write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding='utf8')
    print(json.dumps(result,ensure_ascii=False,indent=2))
    print('SHA256:',hashlib.sha256(out.read_bytes()).hexdigest())
    return result
if __name__=='__main__':main()

resultado = main()

### Salvar relatório no Drive oficial
O código abaixo cria **apenas uma pasta nova** dentro de `results/`, e somente se houve resposta do diagnóstico. Nenhum output antigo é substituído.

In [ ]:
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from datetime import datetime,timezone
api=build("drive","v3")
parent="1iZKAeZPEb6JOlH8KfqsR6-wqI1ygmojI"
folder=api.files().create(body={"name":"EXP-03C_diagnostico_archive_"+datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"),"mimeType":"application/vnd.google-apps.folder","parents":[parent]},fields="id,name").execute()
p="/content/EXP03C_archive_probe/diagnostico_archive.json"
a=api.files().create(body={"name":"diagnostico_archive.json","parents":[folder["id"]]},media_body=MediaFileUpload(p,mimetype="application/json"),fields="id,name").execute()
print("Pasta EXP-03C:","https://drive.google.com/drive/folders/"+folder["id"])

In [ ]:
os.environ.pop("ETHEREUM_ARCHIVE_RPC",None)
del endpoint
print("Variável de endpoint removida do processo. Recomenda-se encerrar o runtime.")